In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import os
import sys
import glob

In [ ]:
STAGATE_info=pd.read_csv('/data/work/file/sample_type.csv')
print(STAGATE_info['Region'].unique())
STAGATE_info_new=STAGATE_info[['sample','selected_res']]
sample_res=STAGATE_info_new.drop_duplicates(subset=['sample','selected_res'], keep='first')
sample_res_dict=dict(zip(sample_res['sample'],sample_res['selected_res']))

In [5]:
def filter_gene(gene_csv):
    bcell_genes = pd.read_csv('/data/work/file/total.Bcell.gene', header=None, names=['gene'])
    bcell_gene_list = bcell_genes['gene'].tolist()
    filtered_gene_csv = gene_csv[~gene_csv['names'].isin(bcell_gene_list)]
    return filtered_gene_csv

In [6]:
def jaccard_index(a, b):
    intersection = len(set(a).intersection(set(b)))
    union = len(set(a).union(set(b)))
    return intersection / union

In [ ]:
for top_n in [50,100,200,300,400,500]:
    all_top_gene={}
    for sample,selected_res in sample_res_dict.items():
        gene_dir=f'/data/work/STAGATE/result/{sample}/tissue_bin100/res{selected_res}/*.csv'
        cluster_list=glob.glob(gene_dir)
        for cluster_file in cluster_list:
            cluster_name = os.path.splitext(os.path.basename(cluster_file))[0]
            top_gene_key=f'{sample}:res{selected_res}:{cluster_name}'
            cluster_csv=pd.read_csv(cluster_file)
            if 'IG' not in cluster_name:
                cluster_csv=filter_gene(cluster_csv)
            cluster_csv = cluster_csv.sort_values(by="logfoldchanges", ascending=False)
            top_gene_value=cluster_csv.loc[:,'names'][0:top_n].tolist()
            all_top_gene[top_gene_key]=top_gene_value
        
    max_len = max(len(lst) for lst in all_top_gene.values())
    for key in all_top_gene.keys():
        if len(all_top_gene[key]) < max_len:
            all_top_gene[key].extend([np.nan] * (max_len - len(all_top_gene[key])))
    all_top_gene_df = pd.DataFrame(all_top_gene)
    all_top_gene_df.to_csv(f'/data/work/STAGATE/heatmap/top{top_n}_genes.csv',index=False)

    t_all_top_gene_df=all_top_gene_df.T
    heatmap_df = pd.DataFrame(index=t_all_top_gene_df.index, columns=t_all_top_gene_df.index)
    for i in t_all_top_gene_df.index:
        tmp = all_top_gene_df[i]
        heatmap_df[i] = [jaccard_index(tmp, all_top_gene_df[x]) for x in t_all_top_gene_df.index]
    np.fill_diagonal(heatmap_df.values, np.nan)
    heatmap_df.to_csv(f'/data/work/STAGATE/heatmap/top{top_n}_heatmap_jaccard.csv')